In [2]:
def topx_indexes(dataframe, nclasses):
    grouped_counts = dataframe.groupby("APP").size()
    grouped_counts = grouped_counts.sort_values(ascending=False)

    topx_groups = grouped_counts.head(nclasses).index

    return topx_groups

def QUIC_dataset(nclasses = 0):
    from cesnet_datazoo.datasets import CESNET_QUIC22
    from cesnet_datazoo.config import DatasetConfig, AppSelection, ValidationApproach

    dataset = CESNET_QUIC22("~/datasets/CESNET-QUIC22/", size="XS")

    common_params = {
        "dataset" : dataset,
        "apps_selection" : AppSelection.ALL_KNOWN,
        "test_period_name" : "W-2022-44",
        "val_approach": ValidationApproach.SPLIT_FROM_TRAIN,
        "train_val_split_fraction": 0.2
    }

    dataset_config = DatasetConfig(**common_params)
    dataset.set_dataset_config_and_initialize(dataset_config)
    train_dataframe = dataset.get_train_df(flatten_ppi=True)
    val_dataframe = dataset.get_val_df(flatten_ppi=True)
    test_dataframe = dataset.get_test_df(flatten_ppi=True)

    if nclasses != 0:
        topx_groups = topx_indexes(train_dataframe, nclasses)

        train_dataframe = train_dataframe[train_dataframe["APP"].isin(topx_groups)]
        test_dataframe  = test_dataframe[test_dataframe["APP"].isin(topx_groups)]
        val_dataframe   = val_dataframe[val_dataframe["APP"].isin(topx_groups)]

    return (train_dataframe, val_dataframe, test_dataframe)

(train_dataframe, val_dataframe, test_dataframe) = QUIC_dataset()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/cesnet_datazoo/config.py:341: UserWarning: Some test dates (20221031) are before or equal to the last train date (20221106). This might lead to improper evaluation and should be avoided.
  warnings.warn(f"Some test dates ({min(test_dates).strftime('%Y%m%d')}) are before or equal to the last train date ({max(train_dates).strftime('%Y%m%d')}). This might lead to improper evaluation and should be avoided.")


Loading data from dataloader


100%|██████████| 8162/8162 [00:34<00:00, 236.37it/s]


Loading data from dataloader


100%|██████████| 192/192 [00:07<00:00, 24.87it/s]


Loading data from dataloader


100%|██████████| 957/957 [00:10<00:00, 89.89it/s] 


In [30]:
from sklearn.ensemble import RandomForestClassifier

X = train_dataframe.drop(columns="APP").to_numpy()
y = train_dataframe["APP"].to_numpy()

X_test = test_dataframe.drop(columns="APP").to_numpy()[:100000]
y_test = test_dataframe["APP"].to_numpy()[:100000]

num_of_sizes = 6
start_index = 66
class_size_sums = {}
for class_i in range(X.shape[1]):
    class_size_sums[class_i] = []
    for size_i in range(num_of_sizes):
        class_size_sums[class_i].append(0)

clf = RandomForestClassifier(max_depth = 10, n_jobs = -1)
clf.fit(X_test, y_test)

def calculate_distance(sum, value, n):
    if n == 0:
        return value

    avg = sum / n

    # if n % 10000 == 0:
    #     print(f"{avg}:{value}")

    return abs(avg - value)

correct = 0
incorrect = 0

ncorrect = 0
nincorrect = 0

arr = [0, 0, 0, 0]

for index, el in enumerate(X):
    if (index + 1) % 100000 == 0:
        break
    sum = 0

    for i in range(num_of_sizes):
        distance = calculate_distance(class_size_sums[y[index]][i],
                                        el[i+start_index],
                                        index)

        sum += distance

        # if index % 10000 == 0:

        # print(f"{i}:{str(int(distance)).ljust(10)}", end = " ")

        class_size_sums[y[index]][i] += el[i+start_index]
    
    # # if index % 10000 == 0:
    centroid = sum / num_of_sizes
  
    if centroid < 50:
        arr[0] += 1
    elif centroid < 200:
        arr[1] += 1
    elif centroid < 400:
        arr[2] += 1
    else:
        arr[3] += 1

    # print(centroid)

    # print(clf.predict(X[index].reshape(1, -1))[0] == y[index], end = " ")

    print(sum / num_of_sizes)
    if index > 99000:
        if clf.predict(X[index].reshape(1, -1))[0] == y[index]:
            print("True")

            correct += sum / num_of_sizes
        
            ncorrect += 1
        else:
            print("False")

            incorrect += sum / num_of_sizes

            nincorrect += 1
print(arr)

print(f"{correct/ncorrect}:{incorrect/nincorrect}")
    


79.66666666666667
164.66666666666666
235.16666666666666
142.16666666666666
294.3333333333333
885.1666666666666
52.333333333333336
212.9047619047619
1350.0
249.35185185185182
52.333333333333336
95.33333333333333
618.0
155.66666666666666
830.0
645.1666666666666
77.20833333333333
449.3333333333333
251.5
44.0
239.7583333333333
584.5
286.84090909090907
749.2753623188406
690.3125
78.04666666666667
41.44871794871795
39.913580246913575
38.654761904761905
724.8103448275862
650.0277777777778
271.6344086021506
148.66666666666666
494.67676767676784
640.2941176470588
189.66666666666666
270.9166666666667
41.76126126126126
480.9956140350877
122.68803418803418
549.0666666666667
94.83333333333333
243.36507936507937
696.9341085271318
710.2310606060605
419.60740740740744
34.21014492753623
587.1666666666666
421.0104166666667
41.5
320.0166666666667
577.4869281045751
107.5
40.11635220125786
53.833333333333336
243.0
339.0803571428571
397.8333333333333
75.41666666666667
50.833333333333336
47.208333333333336
6